# 추론과 제출 파일 생성

학습한 체크포인트로 test 이미지를 예측하여 Kaggle 제출 파일을 만듭니다.

## 1. 설정 변수

- `CONFIG_NAME`: 학습에 쓴 설정 이름.
- `CHECKPOINT`: `"best"`, `"last"`, 또는 `"step001000"` 같은 주기 체크포인트 이름.

In [ ]:
CONFIG_NAME = "config002"
CHECKPOINT = "best"

## 2. 설정 비교

현재 설정 파일과 학습 당시의 사본을 비교합니다. 다르면 다른 키를 알려 주고, 모델은 학습 당시의 사본으로 만듭니다.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from scripts import config as config_utils
from scripts import datautils, evalutils, kaggleutils, network, trainer
from scripts.tokenizer import SmilesTokenizer

config = config_utils.load_snapshot(CONFIG_NAME)
differences = config_utils.diff_config(config, config_utils.load_config(CONFIG_NAME))
if differences:
    print(f"경고: 설정 파일이 학습 당시와 다릅니다: {differences}")
    print("모델은 학습 당시의 설정으로 만듭니다.")
else:
    print("설정이 학습 당시와 같습니다.")

## 3. 모델과 체크포인트 불러오기

저장된 어휘로 토크나이저를 복원하고 모델에 체크포인트를 불러옵니다. 검증 지표는 `best`에만 기록되어 있습니다.

In [ ]:
device = trainer.get_device()
amp = config["train"]["amp"]

tokenizer = SmilesTokenizer.load(config_utils.vocab_path(CONFIG_NAME))
checkpoint = trainer.load_checkpoint(CONFIG_NAME, CHECKPOINT)
model = network.build_model(config, tokenizer, load_pretrained=False)
model.load_state_dict(checkpoint["model"])
model.to(device)

saved_metrics = checkpoint.get("metrics", {})
print(f"체크포인트 {CHECKPOINT}: step {checkpoint['step']}")
print(f"저장 당시 검증 지표: {saved_metrics or '기록 없음'}")

## 4. 검증 점검

검증 세트를 다시 예측하여 저장 당시의 지표와 비교합니다. 혼합 정밀도 때문에 끝자리가 조금 다를 수 있습니다.

In [ ]:
split = datautils.load_split(config_utils.split_path(CONFIG_NAME))
_, val_loader = datautils.build_dataloaders(config, tokenizer, split)
metrics, _ = trainer.evaluate(model, val_loader, tokenizer, device, amp)
display(pd.DataFrame({"저장 당시": pd.Series(saved_metrics, dtype=float), "다시 계산": pd.Series(metrics)}))

## 5. test 예측

test 이미지의 SMILES를 예측합니다. 순서는 `sample_submission.csv`와 같습니다.

In [ ]:
test_loader = datautils.build_test_loader(config)
predictions = trainer.predict_smiles(model, test_loader, tokenizer, device, amp)
print(f"예측 {len(predictions)}개")

## 6. 제출 파일 저장

예측을 정규화하여 저장하고 형식을 검사합니다. 오류가 없으면 Kaggle에 제출할 수 있습니다.

Kaggle은 빈 칸을 허용하지 않으므로 유효하지 않은 예측은 현재 설정 파일의 `submission.default_smiles`로 채웁니다. 이 값은 학습한 뒤에 바꾸어도 됩니다.

In [ ]:
image_ids = [item[0] for item in test_loader.dataset.items]
submission_path = config_utils.predictions_dir(CONFIG_NAME) / f"submission_{CHECKPOINT}.csv"
default_smiles = config_utils.load_config(CONFIG_NAME)["submission"]["default_smiles"]
submission = kaggleutils.make_submission(image_ids, predictions, submission_path, default_smiles)

data_dir = Path(config["data"]["data_dir"])
kaggleutils.validate_submission(submission_path, data_dir / datautils.SAMPLE_SUBMISSION_CSV)
print("형식 검사를 통과했습니다.")

## 7. 결과 확인

유효한 예측의 비율과, test 이미지 몇 장의 예측 구조를 봅니다. 기본값으로 채우기 전의 모델 출력을 기준으로 하며, 유효하지 않은 예측은 "invalid"로 표시합니다.

In [ ]:
num_valid = sum(kaggleutils.normalize_smiles(smiles) != "" for smiles in predictions)
print(f"유효 예측 비율: {num_valid / len(predictions):.3f}")

shown = submission.head(4)
image_dir = data_dir / datautils.TEST_IMAGE_DIR
fig = evalutils.plot_structures(
    [image_dir / f"{image_id}.png" for image_id in shown["image_id"]],
    {"prediction": predictions[:len(shown)]},
)
plt.show()